In [0]:
sales_src_stream=spark.readStream.format("cloudFiles").option("cloudFiles.format","csv").option("cloudFiles.schemaLocation","/Volumes/workspace/ibm/databricks_capstone/schema/sales").option("header",True).load("/Volumes/workspace/ibm/databricks_capstone/input")


sales_src_stream.printSchema()




In [0]:
sales_src_stream.writeStream.format("delta").outputMode("append").option("checkpointLocation","/Volumes/workspace/ibm/databricks_capstone/checkpoint").trigger(availableNow=True).toTable("workspace.ibm.capstone_bronze_sales")

In [0]:
from pyspark.sql.functions import *

sales_src_dataset=spark.read.table("workspace.ibm.capstone_bronze_sales")

print("--------------------------")
print("Data Quality Report")
print("--------------------------")
print("CUSTOMER_ID_NULL: ",sales_src_dataset.filter(col("customer_id").isNull()).count())
print("QUANTITY_LT_0: ",sales_src_dataset.filter(col("quantity").cast("int")<=0).count())
print("NET_AMOUNT_LT_0: ",sales_src_dataset.filter(col("net_amount").cast("double")<=0).count())
print("PRODUCT_ID_UNKNOWN: ",sales_src_dataset.filter(col("product_id")=='UNKNOWN').count())


In [0]:
sales_src_data_with_quality_flag= sales_src_dataset.withColumn("quality_flag",
                          when(col("customer_id").isNull(),"INVALID_CUSTOMER")\
                         .when(col("quantity").cast("int")<=0,"INVALID_QUANTITY")\
                         .when(col("net_amount").cast("double")<=0,"INVALID_AMOUNT")\
                         .when(col("product_id")=='UNKNOWN',"INVALID_PRODUCT")\
                         .otherwise("VALID")
                          )

print("VALID DATA COUNT: ", sales_src_data_with_quality_flag.filter(col("quality_flag")=="VALID").count())

In [0]:
# sales_src_data_with_quality_flag.cache()

## valid data go to silver
sales_src_data_with_quality_flag.filter(col("quality_flag")=="VALID").write.format("delta").mode("overwrite").saveAsTable("workspace.ibm.capstone_silver_sales")

##invalid data go to quarintine
sales_src_data_with_quality_flag.filter(col("quality_flag")!="VALID").write.format("delta").mode("overwrite").saveAsTable("workspace.ibm.capstone_quarantine")

# sales_src_data_with_quality_flag.unpersist()



In [0]:
quarantine_sales_data=spark.read.table("workspace.ibm.capstone_quarantine")
print("---------------------------")
print("Invalid Dataset")
print("---------------------------")
quarantine_sales_data.show()